In [3]:
from IPython.display import Markdown , display 

display(Markdown("## 💥Data Preparation... For Research AI Agent")) 



## 💥Data Preparation... For Research AI Agent

In [ ]:
import json 
import logging
import random 
from pathlib import Path 
from datasets import load_dataset 
import os
from tqdm import tqdm 

OPENHERMES = "teknium/OpenHermes-2.5" 
ALPACA = "yahma/alpaca-cleaned" 
NOROBOT="HuggingFaceH4/no_robots" 

log = logging.getLogger(__name__)
class Data_Preparation: 
    def __init__(self):
        self.output_path = Path("dataset/d1_instruction.jsonl")  
        os.makedirs(self.output_path.parent, exist_ok=True)

    def collect(self): 
        samples = []
        try: 
            print("Loading the OPENHERMES Dataset....")  
            display(Markdown("#### 🔽 Steps ... "))
            ds = load_dataset(OPENHERMES , split="train" , streaming=True)  
            count = 0
            for row in tqdm(ds ,total=20000 ,  desc="OpenHermes Processing"): 
                if count >= 20000: 
                    break 
                if row.get("conversations"): 
                    sample = self._parse_opnhermes(row)   
                    if sample: 
                        samples.append(sample) 
                        count+=1   
                    
            log.info(f"Successfully Loaded the {count} data")
        except Exception as e: 
            log.warning(f"error : {e} aa gaya bhai during processing the openhermes")

        try: 
            print("Loading the ALPACA Dataset....") 
            ds = load_dataset(ALPACA , split="train") 
            c = 0   
            for row in tqdm(ds , desc="Processing Alpaca...."): 
                sample = self._parse_alpaca(row)   
                if sample: 
                    samples.append(sample)  
                    c+=1
                    
                        
            log.info(f"From Alpaca we loaded {c}k rows of data")
        except Exception as e: 
            log.warning(f"error {e} aa gaya bhai during processing the Alpaca") 

        try: 
            print("Loading the no_robots Dataset....") 
            ds = load_dataset(NOROBOT , split="train")  
            count = 0
            for row in tqdm(ds , desc="Processing no_robot.."): 
                sample = self._parse_norobot(row)   
                if sample: 
                    samples.append(sample) 
                    count+=1 
            
            log.info(f"Successfully Loaded the {count} data")
        except Exception as e: 
            log.warning(f"error aa gaya bhai during processing the no_robot error : {e}")

        random.shuffle(samples) 
        self._write(samples) 
        log.info(f"✅ Dataset 1 saved: {len(samples)} samples → {self.output_path}")
        return 

    def _parse_opnhermes(self , row:dict): 
        convs = row.get("conversations" , []) 
        if len(convs)<2: 
            None 

        system = "" 
        messages = [] 
        for row in convs: 
            role = row.get("from" , "").strip()
            value =  row.get("value" , "").strip() 

            if not value: 
                continue 

            if role=="system": 
                system=value
            elif role in ["human" , "user"]: 
                messages.append({"role":"user" , "content":value}) 
            elif role in ['gpt',"assistant"]: 
                messages.append({"role":"assistant" , "content":value})  
        if not messages: 
            return   None 
        
        return {
            "dataset": "openhermes" , 
            "task": "instruction_tune" , 
            "system": system or "You Are Helpful assistant." , 
            "messages":messages 
        }
 
    def _parse_alpaca(self , row:dict): 
        instruction = row.get("instruction" , "").strip() 
        input_text = row.get("input" , "").strip()
        response = row.get("output" , "").strip() 

        if not instruction or not response: 
            return None 

        if len(response) <4: 
            return None 

        user_content = instruction 
        if input_text: 
            user_content = f"{instruction}\n\ninput:\n{input_text}" 

        return {
            "dataset":"alpaca" , 
            "task" : "instruction_tune" , 
            "system" : "You are Helpful assistant." , 
            "messages":[
                {"role":"user" , "content":user_content} , 
                {"role":"assistant" , "content":response}
            ]
        } 
 
    def _parse_norobot(self , row): 
        messages = row.get("messages","") 
        if not messages: 
            return None 

        system = "" 
        message_content = []
        for val in messages: 
            role = val.get("role" , "") 
            value = val.get("content" , "") 

            if role=="system": 
                system = value 
            elif role=="user": 
                message_content.append({"role":"user" , "content":value})
            elif role in ("system" , "assistant"): 
                message_content.append({"role":"assistant" , "content":value}) 
        
        if len(message_content)<2:
            return None 

        return {
            "dataset":"no_robot" , 
            "task":"instruction_tune" , 
            "system" : system or "You are Helpful assistant." , 
            "messages" : message_content
         }


    def _write(self , sample:list): 
        with open(self.output_path , "w") as f: 
            for s in sample: 
                f.write(json.dumps(s , ensure_ascii=False)+"\n") 

        log.info(f"all data is written to {self.output_path}")
 

In [33]:
import shutil
import os

if os.path.isdir("dataset/d1_instruction.jsonl"):
    shutil.rmtree("dataset/d1_instruction.jsonl")

In [34]:
obj = Data_Preparation() 
obj.collect()

Loading the OPENHERMES Dataset....


#### 🔽 Steps ... 

OpenHermes Processing: 100%|██████████| 20000/20000 [16:41<00:00, 19.97it/s]


Loading the ALPACA Dataset....


Processing Alpaca....: 100%|██████████| 51760/51760 [00:01<00:00, 31703.77it/s]


Loading the no_robots Dataset....


Processing no_robot..: 100%|██████████| 9500/9500 [00:00<00:00, 15516.16it/s]


In [ ]:
import json

data = []

with open("dataset/d1_instruction.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

data[:2]

[{'dataset': 'openhermes',
  'task': 'instruction_tune',
  'system': 'You Are Helpful assistant.',
  'messages': [{'role': 'user',
    'content': 'Determine whether the following series converges or diverges: Σ(1/n^2) from n=1 to infinity. Explain your reasoning.'},
   {'role': 'assistant',
    'content': 'The series Σ(1/n^2) from n=1 to infinity converges. This is because it is a p-series with p = 2, which is greater than 1. In general, a p-series Σ(1/n^p) converges if p > 1 and diverges if p ≤ 1. This result can be proven using the integral test or by comparing the series to another known convergent series.\n\nIn this case, since p = 2 > 1, the series converges. This particular series is also known as the Basel problem, and its sum converges to (π^2)/6.'}]},
 {'dataset': 'alpaca',
  'task': 'instruction_tune',
  'system': 'You are Helpful assistant.',
  'messages': [{'role': 'user',
    'content': 'Describe the setting of an enchanted forest.'},
   {'role': 'assistant',
    'content'

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset/d1_instruction.jsonl"
)

dataset.push_to_hub("Rohit-Katkar2003/instruction-dataset")

: 

## Summarization dataset

In [ ]:
SUMMARY_PROMPTS =  [
    "Summarize the following text concisely:\n\n{text}",
    "Extract the key findings and main points from this text:\n\n{text}",
    "Write a brief, accurate summary suitable for a research report:\n\n{text}",
    "What are the most important takeaways from this content?\n\n{text}",
    "Provide a dense, informative summary of the following:\n\n{text}",
]

SYSTEM_PROMPT = (
    "You are a research summarization assistant. "
    "Your summaries are accurate, concise, and preserve all key facts."
)

class Prep_Summarization_data: 
    def __init__(self): 
        self.output_path = Path("dataset/d2_summarization.jsonl") 
        os.makedirs(self.output_path.parent , exist_ok=True) 

    def collect(self): 
        display(Markdown("## 🎈Loading Summarization dataset....")) 

        try: 
            count = 40000 ## taking 40000 rows only 
            ds = load_dataset("abisee/cnn_dailymail" , split="train" , streaming=True)  
            indices = random.sample(range(len(ds)) , min(20000 , len(ds)))  # it return list
            for i in indices: 
                 
                
        except Exception as e: 
            print((f"got error : {e}"))
            log.warning(f"got error : {e}")